<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Building Vectors from PDFs
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

## Using Unstructured.io to parse the pdf file into a table

In [1]:
# Unstructured libraries
from unstructured.partition.image import partition_image
from unstructured.partition.pdf import partition_pdf

# For the Embeddigns
import sys
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
import pandas as pd
import json

# Teradata Vector Store libraries
from getpass import getpass
from teradatagenai import VSManager, VectorStore, VSPattern, VSApi
from teradataml import create_context, set_auth_token, execute_sql, display, DataFrame
display.max_rows = 100000

import numpy as np
import re


In [3]:
#Partition the PDF file into chunks
pdfpath = r"DensePassageRetrieval.pdf"

# Partition the PDF document
elements = partition_pdf(
    filename=pdfpath,                  # mandatory
    strategy="hi_res",                                     # mandatory to use ``hi_res`` strategy
    extract_images_in_pdf=True,                            # mandatory to set as ``True``          
    extract_image_block_to_payload=False,                  # optional
    )

In [4]:
#Convert the list of elements to a list of dictionaries
element_dicts = [element.to_dict() for element in elements]

# save the list locally:
file = r"UnstructuredDemoJSON.json"
with open(file, "w") as file:
    json.dump(element_dicts, file, indent=2)

In [5]:
for element in elements:
    print(element)

0
2020
2
0
2
p e S 0 3 ] L C . s c [ 3 v 6 0 9 4 0 . 4 0 0 2
:
v
i
X
r
a
Dense Passage Retrieval for Open-Domain Question Answering
Vladimir Karpukhin∗, Barlas O˘guz∗, Sewon Min†, Patrick Lewis, Ledell Wu, Sergey Edunov, Danqi Chen‡, Wen-tau Yih
Facebook AI †University of Washington ‡Princeton University
{vladk, barlaso, plewis, ledell, edunov, scottyih}@fb.com sewon@cs.washington.edu danqic@cs.princeton.edu
Abstract
Open-domain question answering relies on ef- ﬁcient passage retrieval to select candidate contexts, where traditional sparse vector space models, such as TF-IDF or BM25, are the de facto method. In this work, we show that retrieval can be practically implemented us- ing dense representations alone, where em- beddings are learned from a small number of questions and passages by a simple dual- encoder framework. When evaluated on a wide range of open-domain QA datasets, our dense retriever outperforms a strong Lucene- BM25 system greatly by 9%-19% absolute in terms of top-20

## Connect to Vantage

In [8]:
# Connect to Vantage using create_context.
hostname = getpass(prompt = 'hostname: ')
username = getpass(prompt = 'username: ')
password = getpass(prompt = 'password: ')

context=create_context(host=hostname, username=username, password=password)

## Load the chunked text into SQL

In [27]:
execute_sql("""
CREATE MULTISET TABLE pdf_elements (
    id INTEGER GENERATED ALWAYS AS IDENTITY NOT NULL,
    text VARCHAR(10000),
    PRIMARY KEY (id)
);
""")

TeradataCursor uRowsHandle=525 bClosed=False

In [24]:
#Clean data function
import re
def clean_data(data):
    # Remove non-ASCII characters
    cleanData = re.sub(r'[^\x00-\x7F]+', '', data)
    return cleanData.replace('·', '*').replace("'", "")

In [28]:
# Insert the elements into the Teradata table
for element in elements:
    if element.text:
        print(clean_data(element.text))
        execute_sql(f"""
        INSERT INTO pdf_elements (text) VALUES ('{clean_data(element.text)}');
        """)

0
2020
2
0
2
p e S 0 3 ] L C . s c [ 3 v 6 0 9 4 0 . 4 0 0 2
:
v
i
X
r
a
Dense Passage Retrieval for Open-Domain Question Answering
Vladimir Karpukhin, Barlas Oguz, Sewon Min, Patrick Lewis, Ledell Wu, Sergey Edunov, Danqi Chen, Wen-tau Yih
Facebook AI University of Washington Princeton University
{vladk, barlaso, plewis, ledell, edunov, scottyih}@fb.com sewon@cs.washington.edu danqic@cs.princeton.edu
Abstract
Open-domain question answering relies on ef- cient passage retrieval to select candidate contexts, where traditional sparse vector space models, such as TF-IDF or BM25, are the de facto method. In this work, we show that retrieval can be practically implemented us- ing dense representations alone, where em- beddings are learned from a small number of questions and passages by a simple dual- encoder framework. When evaluated on a wide range of open-domain QA datasets, our dense retriever outperforms a strong Lucene- BM25 system greatly by 9%-19% absolute in terms of top-20 passage

## Create embeddings in SQL for the chunked pdf

In [29]:
#Create embeddings from the pdf
execute_sql("""
CREATE TABLE pdf_embeddings AS (
    SELECT * FROM AI_TextEmbeddings(
        ON pdf_elements AS InputTable
        USING
        authorization(AWSEmbeddingsAuth)
        region('us-west-2')
        apitype('aws')
        modelname('amazon.titan-embed-text-v1')
        textcolumn('text')
        outputformat('vector')
    ) as DT
) WITH DATA;   
""")

TeradataCursor uRowsHandle=727 bClosed=False

In [26]:
#Drop the table
execute_sql("""
DROP TABLE pdf_elements;
""")

TeradataCursor uRowsHandle=524 bClosed=False